**Tiny MathNet Grokking Encoder Summary**

This notebook summarizes the from-scratch MathNet text classifier experiment, the training runs, the monitoring results, the final test results, and the main conclusions.

Run directory: `runs/tiny_mathnet_grokking`

Main goal: train a sub-1M parameter encoder from scratch on MathNet problem text only, using a train-only BPE tokenizer, MLM pretraining, supervised classification, joint training, and a long grokking-style continuation.


**What Was Implemented**

- Local Python environment with PyTorch MPS, tokenizers, MLX, NumPy, scikit-learn, pandas, matplotlib, psutil, and tqdm.
- CLI commands for `prepare`, `benchmark`, `pretrain`, `finetune`, `joint`, `long-joint`, `evaluate`, and `report`.
- Train-only BPE tokenizer with vocab size 4096 and special tokens `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`.
- Cached encoded train, validation, and test arrays with token IDs, attention masks, labels, and original dataset indices.
- Encoder-only Transformer with random initialization, padding-aware attention, `[CLS]` classification, MLM head, and tied output projection behavior where practical.
- Device-resident batches for the main training path, with training, masking, forward pass, backward pass, and optimizer step on MPS.
- Hardware and correctness checks, including MPS availability, throughput benchmark, tiny overfit test, MLM loss-decrease test, checkpointing, and full test evaluation.
- Duplicate-group split support was added as `prepare --group-duplicates`, but the main long run used the original stratified split.


**Dataset And Split**

| item | value |
|---|---:|
| Total labeled examples | 27231 |
| Train examples | 21784 |
| Validation examples | 2723 |
| Test examples | 2724 |
| BPE vocab size | 4096 |
| Max sequence length | 256 |

| Category | Total | Train | Validation | Test |
|---|---:|---:|---:|---:|
| Algebra | 7859 | 6287 | 786 | 786 |
| Combinatorics | 5864 | 4691 | 586 | 587 |
| Geometry | 8313 | 6650 | 831 | 832 |
| Number Theory | 5195 | 4156 | 520 | 519 |

Leakage controls: the data was split before tokenizer fitting, the tokenizer was trained only on the training split, MLM pretraining used only training split problem text, validation selected checkpoints, and the test split was evaluated only after model selection.

Caveat: the original split is not duplicate-clean. A normalized duplicate scan found 85 duplicate groups and 172 rows crossing train, validation, and test, about 0.6 percent of the labeled dataset. Twelve duplicate groups had more than one label. This is a dataset caveat, not a tokenizer or training-loop leakage issue.


**Model Configuration**

| setting | value |
|---|---:|
| Parameters | 656324 |
| Vocab size | 4096 |
| Max length | 256 |
| Model width | 96 |
| Layers | 3 |
| Attention heads | 4 |
| Feedforward width | 192 |
| Dropout | 0.15 |
| Classes | 4 |

The model stayed under the 1M parameter constraint and trained comfortably under 8 GB of RAM. The long run process RSS stayed low, while MPS driver allocation was stable for the workload.


**Training Runs**

| Phase | Purpose | Main result |
|---|---|---|
| Supervised | Direct classifier from scratch | Good baseline, test macro F1 0.8475 |
| MLM pretrain | From-scratch masked language modeling on train problems | MLM validation loss decreased, masked accuracy reached about 0.226 |
| Finetune | Classification after MLM pretraining | Test macro F1 0.8439, not better than direct supervised |
| Joint | Classification plus MLM continuation | Best overall short-run test macro F1 0.8490 |
| Long joint | 200-epoch grokking-style continuation from joint best | No grokking, final validation lower than early best |

The long continuation used AdamW, cosine schedule, high weight decay, label smoothing, MLM loss, token dropout, and math-token masking. It ran as a single saturated MPS job with periodic checkpoints and frequent metric logging.


**Long Run Monitoring**

| metric | value |
|---|---:|
| Epochs | 200 |
| Runtime seconds | 10321.54 |
| Runtime hours | 2.87 |
| Average tokens per second | 112199.05 |
| Min tokens per second | 84388.94 |
| Max tokens per second | 124429.58 |
| Final tokens per second | 123205.08 |

| Long run point | Epoch | Train loss | Validation loss | Validation accuracy | Validation macro F1 |
|---|---:|---:|---:|---:|---:|
| Best validation | 3 | 1.5393 | 0.4113 | 0.8641 | 0.8541 |
| Final | 200 | 1.2968 | 0.5195 | 0.8498 | 0.8394 |

The run was technically healthy. There were no NaNs, no memory runaway, and no evidence of training-loop tokenization or CPU starvation. Throughput varied, but recovered repeatedly and finished near the top of the observed range.


**Test Results**

| Checkpoint | Accuracy | Macro F1 | Loss |
|---|---:|---:|---:|
| Supervised best | 0.8568 | 0.8475 | 0.4185 |
| MLM finetune best | 0.8535 | 0.8439 | 0.4225 |
| Joint best | 0.8594 | 0.8490 | 0.4250 |
| Long joint best | 0.8590 | 0.8485 | 0.4331 |

The best short joint checkpoint slightly outperformed the long-joint best checkpoint on test macro F1. The far longer run did not improve held-out performance.


**Long-Joint Full Test Set By Category**

| True category | Correct | Total | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| Algebra | 657 | 786 | 0.8359 | 0.8544 | 0.8359 | 0.8450 |
| Combinatorics | 489 | 587 | 0.8330 | 0.7641 | 0.8330 | 0.7971 |
| Geometry | 793 | 832 | 0.9531 | 0.9612 | 0.9531 | 0.9572 |
| Number Theory | 401 | 519 | 0.7726 | 0.8184 | 0.7726 | 0.7948 |

Confusion matrix rows are true labels and columns are predicted labels, ordered as Algebra, Combinatorics, Geometry, Number Theory.

| True label | Algebra | Combinatorics | Geometry | Number Theory |
|---|---:|---:|---:|---:|
| Algebra | 657 | 64 | 11 | 54 |
| Combinatorics | 48 | 489 | 16 | 34 |
| Geometry | 12 | 26 | 793 | 1 |
| Number Theory | 52 | 61 | 5 | 401 |

Category accuracy differed strongly. Geometry was easiest, while Number Theory was weakest. A simple correct-versus-incorrect category chi-square gave p = 1.92e-21.


**Why Geometry Was Easier**

The strongest theory is lexical separability. Geometry problems often announce themselves with highly diagnostic tokens such as triangle, circle, tangent, circumcircle, incircle, angle, bisector, altitude, and orthocenter.

| Category | Model accuracy | Geometry-cue words in test problems |
|---|---:|---:|
| Algebra | 0.8359 | 0.028 |
| Combinatorics | 0.8330 | 0.237 |
| Geometry | 0.9531 | 0.804 |
| Number Theory | 0.7726 | 0.081 |

A simple TF-IDF logistic regression baseline reached 0.9351 accuracy on Geometry, close to the Transformer result of 0.9531. That suggests much of the Geometry advantage comes from category-specific surface language, not deep geometric reasoning.

The model did not use the `images` metadata field. Half of Geometry examples had image metadata, but only 7.6 percent of Geometry test problems had direct image or URL markup in `problem_markdown`, so images are probably not the main explanation.


**Grokking Conclusion**

No grokking-like delayed generalization was observed.

The key evidence is that the long run's best validation macro F1 occurred at epoch 3, while training loss continued to decrease through epoch 200 and validation loss rose. A grokking pattern would usually show a delayed validation jump after a period of memorization or poor generalization. This run showed early best performance followed by a lower plateau.

The long run was still useful. It ruled out the simplest hypothesis that the model only needed much more training time under the chosen objective and schedule.


**Main Takeaways**

- A tiny 656k parameter encoder can learn MathNet category labels from scratch with about 0.86 test accuracy.
- MLM pretraining on the same small dataset did not improve classification in this run.
- Joint classification plus MLM gave the best test macro F1, but only by a small margin.
- A 200-epoch continuation did not produce grokking.
- Geometry is much easier because its text is more category-specific.
- Number Theory is hardest because its wording overlaps heavily with Algebra and some Combinatorics.
- The pipeline avoided tokenizer and pretraining contamination, but the original split has a small duplicate-crossing caveat.
- The next clean experiment should use `prepare --group-duplicates` and rerun the baseline plus joint training on a duplicate-clean split.


In [ ]:
from pathlib import Path
import json

run_dir = Path("runs/tiny_mathnet_grokking")
eval_dir = run_dir / "eval"

for path in sorted(eval_dir.glob("*_test.json")):
    data = json.loads(path.read_text())
    metrics = data["metrics"]
    print(path.name, metrics)
